#### Day 4: Chains - The Pipe

Chain model calls together, run many inputs at once, and stream the answer as it's written.

In [ ]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3-ultra-550b-a55b:free")

In [ ]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001D43F843B10>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={})

#### 1. The two steps a chain replaces

In [ ]:
summarizer = ChatPromptTemplate.from_messages([
    ("system", "You summarize text for busy people."),
    ("user", "Summarizer this in one sentence:\n\n{text}"),
])

article = """The city council approved a plan to plant 12,000 trees across
the eastern districts over the next four years. The scheme is funded by a
levy on new commercial developments, and the first 2,000 trees will go in
along the ring road where air quality readings have been worst."""

messages = summarizer.invoke({"text": article})

model.invoke(messages).text

'The city council approved a four-year plan to plant 12,000 trees in eastern districts—funded by a levy on new commercial developments—starting with 2,000 trees along the high-pollution ring road.'


Two steps: build the messages, then send them. You will write that pair thousands of times, so LangChain lets you join them once.

#### 2. Join a template and a model with the pipe

In [ ]:
chain = summarizer | model
chain.invoke({"text": article}).text

'The city council approved a four-year, levy-funded plan to plant 12,000 trees in eastern districts, starting with 2,000 along the high-pollution ring road.'

summariser | model is one object now. Give it a dictionary, get a reply.

| is Python's or-operator, taught to hand output from the left to the right.

#### 3. Add StrOutputParser to get a plain string

usually want the plain string, not the reply object.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = summarizer | model | StrOutputParser()

chain.invoke({"text": article})

'The city council approved a four-year, developer-funded plan to plant 12,000 trees in eastern districts, starting with 2,000 along the high-pollution ring road.'

No .text at the end, the parser did that.

**Read the chain left to right:** dictionary in, messages out of the template, a reply out of the model, a string out of the parser. Easier than reading nested calls from the inside out.

#### 4. Look inside a chain: its steps and its inputs
A chain is not a black box. It is a list of the things you joined.

In [ ]:
chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarize text for busy people.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Summarizer this in one sentence:\n\n{text}'), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling

In [ ]:
[type(step).__name__ for step in chain.steps]

['ChatPromptTemplate', 'ChatOpenRouter', 'StrOutputParser']

The three objects, in order. It also knows what input it expects, worked out from the template.

In [ ]:
list(chain.get_input_jsonschema()["properties"])

['text']

#### 5. Every piece is a Runnable: invoke, batch, stream
The template, model, parser, and the whole chain share three verbs: .invoke(), .batch(), .stream(). That shared contract is the only reason | works, no type registry, just an agreement on methods.

In [25]:
summarizer.invoke({"text":"hello"})

ChatPromptValue(messages=[SystemMessage(content='You summarize text for busy people.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Summarizer this in one sentence:\n\nhello', additional_kwargs={}, response_metadata={})])

In [27]:
model.invoke("Say hi in three words.")

AIMessage(content='Hello there, friend.', additional_kwargs={'reasoning_content': 'The user wants a greeting in exactly three words.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a greeting in exactly three words.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1792677135-3ovWZUaZp1wOaAzTG0Jl', 'created': 1792677135, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a163e2-4372-715a-a8ac-b5e9ca63bc5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 17, 'total_tokens': 39, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 13}})

In [30]:
from langchain.messages import AIMessage
StrOutputParser().invoke(AIMessage(content='Hello there, friend.', additional_kwargs={'reasoning_content': 'The user wants a greeting in exactly three words.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants a greeting in exactly three words.'}]}, response_metadata={'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'id': 'gen-1792677135-3ovWZUaZp1wOaAzTG0Jl', 'created': 1792677135, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--01a163e2-4372-715a-a8ac-b5e9ca63bc5d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 22, 'output_tokens': 17, 'total_tokens': 39, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 13}}))

'Hello there, friend.'

In [31]:
StrOutputParser().invoke(model.invoke("Say hi in three words."))

'Hello there, friend.'

#### 6. Chain two model calls together
Summarise the article, then turn the summary into a headline. The second template needs a dictionary with a summary key; the first chain produces a plain string, so something has to wrap it.

In [32]:
headline_writer = ChatPromptTemplate.from_messages([
    ("system", "You write short newspaper headlines. Five words at most."),
    ("user", "Write a headline for this summary:\n\n{summary}"),
])

In [33]:
time.sleep(4)

two_sleep = (
    chain
    | (lambda summary: {"summary":summary})
    | headline_writer
    | model
    | StrOutputParser()
)

two_sleep.invoke({"text": article})

'Council Approves 12,000 Tree Plan'


A plain Python function dropped into the pipe, LangChain wraps it for you. The trick to composing these: at each |, check what the left produces and what the right needs, and bridge the gap with a small function.

#### Two chains in a plain Python function
Often you want the summary and the headline, not just the last thing.

In [35]:
time.sleep(4)

def summarize_and_title(inputs):
    summary = chain.invoke(inputs)
    headline = (headline_writer | model | StrOutputParser()).invoke({"summary": summary})
    return {"summary": summary, "headline": headline}

summarize_and_title({"text":article})

{'summary': 'The city council approved a four-year, levy-funded plan to plant 12,000 trees in eastern districts, starting with 2,000 along the high-pollution ring road.',
 'headline': 'Council Approves 12,000 Tree Planting Plan'}

Plain Python, calling two chains. Not everything has to be one long pipe, a readable function beats a clever expression.



#### 7. Run many inputs at once with .batch()
.batch() sends several inputs together rather than one after another.

In [36]:
time.sleep(6)

articles = [
    {"text": "Rainfall in the northern hills was 40 percent above average in July."},
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"text": "Library visits rose for the first time in nine years."},
]

chain.batch(articles)

['Rainfall in the northern hills exceeded the July average by 40 percent.',
 'The new bridge reduced the average commute by 23 minutes, from 55 to 32 minutes.',
 'Library visits increased for the first time in nearly a decade.']

In [37]:
chain

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarize text for busy people.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Summarizer this in one sentence:\n\n{text}'), additional_kwargs={})])
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling


Three summaries in roughly the time of one.

Careful on a free key: .batch() fires calls concurrently, which trips the per-minute limit. You can hold it back.

In [39]:
time.sleep(6)

chain.batch(articles, config={"max_concurrency":1})

['July rainfall in the northern hills exceeded the average by 40 percent.',
 'The new bridge reduced the average commute by 23 minutes, from 55 to 32 minutes.',
 'Library visits increased for the first time in nearly a decade.']

max_concurrency: 1 runs them one at a time. Slower, much less likely to fail when twenty people share the same limit.

#### Keep the good results with return_exceptions
By default, one failing input raises the whole .batch() call, losing the good results too.

In [40]:
mixed = [
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"wrong_key": "this input is missing the text variable"},
]

results = chain.batch(mixed, return_exceptions=True)
results

['The new bridge reduced the average commute by 23 minutes, from 55 to 32 minutes.',
 KeyError("Input to ChatPromptTemplate is missing variables {'text'}.  Expected: ['text'] Received: ['wrong_key']\nNote: if you intended {text} to be part of the string and not a variable, please escape it with double curly braces like: '{{text}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT ")]

In [42]:
time.sleep(6)

mixed = [
    {"text": "The new bridge cut the average commute from 55 minutes to 32."},
    {"wrong_key": "this input is missing the text variable"},
]

results = chain.batch(mixed, return_exceptions=True)

for r in results:
    if isinstance(r, Exception):
        print("failed:" + type(r).__name__)
    else:
        print("ok:" + r)

ok:The new bridge reduced the average commute by 23 minutes, from 55 to 32 minutes.
failed:KeyError


return_exceptions=True puts the error in the list instead of raising it, test which is which with isinstance.

For real data this is usually what you want: process what you can, collect failures for later.

#### 8. Stream the answer with .stream()
.stream() hands you the answer in pieces as it arrives.

In [43]:
chain.stream({"text": article})

<generator object RunnableSequence.stream at 0x000001D43FD92C50>

In [44]:
# time.sleep(6)

for piece in chain.stream({"text": article}):
    print(piece, end="")

The city council approved a four-year, developer-funded plan to plant 12,000 trees in eastern districts, starting with 2,000 along the polluted ring road.

Not faster, the last word arrives at the same moment either way. What changes: the user stops staring at a blank screen, why every chatbot does this.

It streams because every step supports streaming. A step that must see the whole input first, like the wrapping function in section 6, quietly kills streaming before that point.

#### 9. Add a backup model with .with_fallbacks()
Free models fail. .with_fallbacks() lets you name a second one to try.



In [45]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001D43F843B10>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={})

In [46]:
backup = ChatOpenRouter(model="meta-llama/llama-3.3-70b-instruct:free",temperature = 0)

resilient = model.with_fallbacks([backup])

safe_chain = summarizer | resilient | StrOutputParser()

In [47]:
# time.sleep(6)

safe_chain.invoke({"text": article})

'The city council approved a four-year, developer-funded plan to plant 12,000 trees in eastern districts, starting with 2,000 along the polluted ring road.'

If the first model raises, the second is tried with the same input, the chain around it doesn't change. Worth knowing on a free tier, where a 429 is a matter of when, not if.